In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from text_processing import *
from utils import *
hotpot_file_candidates = [
    REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json",
    REPO_ROOT / "hotpot_dev_distractor_v1.json",
]
file_path = next((str(path) for path in hotpot_file_candidates if path.exists()), str(hotpot_file_candidates[0]))
print(f"Using HotpotQA file: {file_path}")
documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=500)
# print("Example document:\n")
# print("Title:", documents[0]["title"])
# print("Text:", documents[0]["text"][:200])

Using HotpotQA file: /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_dev_distractor_v1.json
Loading cached dataset...
Loaded 4937 documents
Loaded 500 samples


In [3]:
for index in range(40):
    print("Question:",samples[index]['question'])
    for idx in samples[index]['gold_doc_ids']:
        print("Document:", documents[idx]['text'])

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Document: Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.
Document: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Document: Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer.  In the film, two teenage girls cause their respective parents much concern when they start to become interested in boys.  The parents' bickering about which girl is the worse influence causes more problems than it solves.
Docum

In [4]:
import RAG_graph

GRAPH_CACHE_PATH = REPO_ROOT / "cache" / "hotpotqa_prefinalize_graph.pkl"
GRAPH_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
GRAPH_BATCH_SIZE = 4
REBUILD_GRAPH_CACHE = False

GRAPH_CONFIG = dict(
    text_embed_dim=1024,
    df_ratio=0.9,
    buffer_size=100,
    chunk_size=256,
    remove_duplicate_token=True,
    device="cuda",
    plot_embeds=False,
)

if REBUILD_GRAPH_CACHE or not GRAPH_CACHE_PATH.exists():
    graph_database = RAG_graph.ProtoGraphRAG(**GRAPH_CONFIG)
    graph_database.index_json(documents, batch_size=GRAPH_BATCH_SIZE)
    graph_database.save_data_split(str(GRAPH_CACHE_PATH))
    print(f"Saved pre-finalize graph cache to {GRAPH_CACHE_PATH}")
else:
    print(f"Using existing pre-finalize graph cache: {GRAPH_CACHE_PATH}")

# Always reload a fresh pre-finalize graph before running finalize().
graph_database = RAG_graph.ProtoGraphRAG.load_data_split(str(GRAPH_CACHE_PATH))
# Modify finalize-related settings on graph_database here before calling finalize().
graph_database.finalize()
graph_database.print_memory_size()


Using existing pre-finalize graph cache: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_prefinalize_graph.pkl
Loading text encoder models in device: GPU


/home/xiaoyue/anaconda3/envs/llm_graph/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[233050.7057s] Finalize started.
[233050.7065s] Computed average chunk length.
finalize_token_nodes: 64900/64900
[233068.7805s] Finished token node finalization.
merge_duplicate_description_proto_nodes: 33/33
[233150.3833s] Finished merging proto nodes by description.
[233150.6762s] Finished assigning token and proto IDF.
[233150.9611s] Finished computing proto BM25.
[233151.5166s] Finished building query database.
[233151.6308s] Finished building phrase query index.
[233151.8189s] Finished building chunk-to-proto edges.
[233151.8223s] Finalizing completed.
memory size: 217.69 MB


In [5]:
graph_database.show_proto_description_logs()

'proto_description_logs.txt'

In [5]:
graph_database.show_multi_proto_token_nodes(min_proto_count=2,
            max_sentences_per_proto=30,
            as_html=True,
            token_contains=None,
            sort_by="proto_count",
            max_token_nodes=50,
            max_protos_per_token=10,
            max_examples_per_token=50,
            open_details=False)

In [6]:
graph_database.show_described_proto_token_nodes(                         max_sentences_per_proto=3,                                           as_html=True,                                                        token_contains=None,                                                 sort_by="proto_count",                                               max_token_nodes=20,                                                  max_protos_per_token=10,                                             max_examples_per_token=10,                                           open_details=False,                                              )

In [7]:
from collections import Counter
from datetime import datetime
import csv
import re

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

PLOT_CACHE_ROOT = REPO_ROOT / "cache" / "proto_description_label_plots"
PLOT_RUN_DIR = PLOT_CACHE_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
PLOT_RUN_DIR.mkdir(parents=True, exist_ok=True)

prefinalize_graph = RAG_graph.ProtoGraphRAG.load_data_split(str(GRAPH_CACHE_PATH))
prefinalize_graph.proto_description_label_contains_text = True
candidate_bank_cache = {}

def _safe_filename(text, max_len=80):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")
    return (safe or "empty")[:max_len]

def _embedding_for_prediction_record(graph, proto_node, prediction_record, retained_lookup):
    span_occurrence = prediction_record["span_occurrence"]
    retained_candidates = retained_lookup.get(graph._get_span_occurrence_key(span_occurrence), [])
    if retained_candidates:
        retained_text_embedding = retained_candidates.pop()
        if retained_text_embedding.embed is not None:
            return retained_text_embedding.embed.detach().cpu().float().numpy()

    regenerated_text_embedding = graph._reencode_text_embedding_for_occurrence(
        proto_node.token_node,
        span_occurrence,
    )
    if regenerated_text_embedding is None or regenerated_text_embedding.embed is None:
        return None
    return regenerated_text_embedding.embed.detach().cpu().float().numpy()

def _plot_labeled_points(coords, labels, title, output_path):
    unique_labels = sorted(set(labels))
    cmap = plt.get_cmap("tab20", max(len(unique_labels), 1))
    plt.figure(figsize=(8, 6), dpi=140)
    for label_index, label in enumerate(unique_labels):
        mask = np.array([item == label for item in labels])
        plt.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=24,
            alpha=0.78,
            label=label,
            color=cmap(label_index),
        )
    plt.title(title)
    plt.xlabel("component 1")
    plt.ylabel("component 2")
    plt.legend(loc="best", fontsize=7, frameon=True)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

target_token_nodes = [
    token_node for token_node in prefinalize_graph.token_nodes
    if len(token_node.proto_node_list) > 1
]

summary_rows = []
point_rows = []

for token_node in target_token_nodes:
    for proto_node in token_node.proto_node_list:
        all_span_occurrences = prefinalize_graph._sample_proto_span_occurrences(proto_node, max_samples=None)
        if len(all_span_occurrences) < 10:
            summary_rows.append({
                "token_text": token_node.token_text,
                "proto_node_id": proto_node.proto_node_id,
                "status": "skipped_fewer_than_10_samples",
                "sample_count": len(all_span_occurrences),
                "label_count": 0,
                "label_counts": {},
                "precheck_label_counts": {},
                "pca_path": "",
                "tsne_path": "",
            })
            continue

        precheck_result = prefinalize_graph._predict_proto_description_from_samples(
            proto_node,
            model=prefinalize_graph.proto_description_model,
            candidate_bank_cache=candidate_bank_cache,
            max_samples=10,
        )
        if precheck_result is None:
            summary_rows.append({
                "token_text": token_node.token_text,
                "proto_node_id": proto_node.proto_node_id,
                "status": "skipped_no_precheck_predictions",
                "sample_count": len(all_span_occurrences),
                "label_count": 0,
                "label_counts": {},
                "precheck_label_counts": {},
                "pca_path": "",
                "tsne_path": "",
            })
            continue

        precheck_label_counts = precheck_result.get("description_counts", {})
        if precheck_result.get("sample_count", 0) < 10:
            summary_rows.append({
                "token_text": token_node.token_text,
                "proto_node_id": proto_node.proto_node_id,
                "status": "skipped_fewer_than_10_precheck_predictions",
                "sample_count": precheck_result.get("sample_count", 0),
                "label_count": len(precheck_label_counts),
                "label_counts": {},
                "precheck_label_counts": precheck_label_counts,
                "pca_path": "",
                "tsne_path": "",
            })
            continue

        if precheck_result.get("top_description_count", 0) >= 8:
            summary_rows.append({
                "token_text": token_node.token_text,
                "proto_node_id": proto_node.proto_node_id,
                "status": "skipped_consensus_8_of_10",
                "sample_count": precheck_result.get("sample_count", 0),
                "label_count": len(precheck_label_counts),
                "label_counts": {},
                "precheck_label_counts": precheck_label_counts,
                "pca_path": "",
                "tsne_path": "",
            })
            continue

        prediction_result = prefinalize_graph._predict_proto_description_from_samples(
            proto_node,
            model=prefinalize_graph.proto_description_model,
            candidate_bank_cache=candidate_bank_cache,
            max_samples=None,
        )
        if prediction_result is None:
            summary_rows.append({
                "token_text": token_node.token_text,
                "proto_node_id": proto_node.proto_node_id,
                "status": "no_predictions",
                "sample_count": 0,
                "label_count": 0,
                "label_counts": {},
                "precheck_label_counts": precheck_label_counts,
                "pca_path": "",
                "tsne_path": "",
            })
            continue

        retained_lookup = prefinalize_graph._build_retained_text_embedding_lookup(proto_node)
        embeddings = []
        labels = []
        kept_records = []

        for record in prediction_result["sample_prediction_records"]:
            embedding = _embedding_for_prediction_record(prefinalize_graph, proto_node, record, retained_lookup)
            if embedding is None:
                continue
            embeddings.append(embedding)
            labels.append(record.get("predicted_description") or "(none)")
            kept_records.append(record)

        label_counts = Counter(labels)
        if len(embeddings) < 2:
            summary_rows.append({
                "token_text": token_node.token_text,
                "proto_node_id": proto_node.proto_node_id,
                "status": "too_few_embeddings",
                "sample_count": len(embeddings),
                "label_count": len(label_counts),
                "label_counts": dict(label_counts),
                "precheck_label_counts": precheck_label_counts,
                "pca_path": "",
                "tsne_path": "",
            })
            continue

        matrix = np.stack(embeddings)
        token_safe = _safe_filename(token_node.token_text)
        base_name = f"token_{token_node.token_node_id}_{token_safe}_proto_{proto_node.proto_node_id}"

        pca_coords = PCA(n_components=2, random_state=42).fit_transform(matrix)
        pca_path = PLOT_RUN_DIR / f"{base_name}_pca.png"
        _plot_labeled_points(
            pca_coords,
            labels,
            f"PCA | token={token_node.token_text!r} | proto={proto_node.proto_node_id}",
            pca_path,
        )

        tsne_perplexity = min(30, max(1, len(matrix) - 1))
        tsne_coords = TSNE(
            n_components=2,
            random_state=42,
            init="random",
            learning_rate="auto",
            perplexity=tsne_perplexity,
        ).fit_transform(matrix)
        tsne_path = PLOT_RUN_DIR / f"{base_name}_tsne.png"
        _plot_labeled_points(
            tsne_coords,
            labels,
            f"TSNE | token={token_node.token_text!r} | proto={proto_node.proto_node_id}",
            tsne_path,
        )

        summary_rows.append({
            "token_text": token_node.token_text,
            "proto_node_id": proto_node.proto_node_id,
            "status": "plotted",
            "sample_count": len(embeddings),
            "label_count": len(label_counts),
            "label_counts": dict(label_counts),
            "precheck_label_counts": precheck_label_counts,
            "pca_path": str(pca_path),
            "tsne_path": str(tsne_path),
        })

        for record, label in zip(kept_records, labels):
            span_occurrence = record["span_occurrence"]
            point_rows.append({
                "token_text": token_node.token_text,
                "proto_node_id": proto_node.proto_node_id,
                "chunk_node_id": span_occurrence.chunk_node.chunk_node_id,
                "span_start": span_occurrence.span_start,
                "span_end": span_occurrence.span_end,
                "span_text": span_occurrence.span_text,
                "predicted_description": label,
                "predicted_label": record.get("predicted_label"),
                "prediction_score": record.get("prediction_score"),
            })

summary_path = PLOT_RUN_DIR / "summary.csv"
with summary_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["token_text", "proto_node_id", "status", "sample_count", "label_count", "label_counts", "precheck_label_counts", "pca_path", "tsne_path"],
    )
    writer.writeheader()
    writer.writerows(summary_rows)

points_path = PLOT_RUN_DIR / "point_labels.csv"
with points_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["token_text", "proto_node_id", "chunk_node_id", "span_start", "span_end", "span_text", "predicted_description", "predicted_label", "prediction_score"],
    )
    writer.writeheader()
    writer.writerows(point_rows)

print(f"Saved proto semantic-label plots to: {PLOT_RUN_DIR}")
print(f"Summary CSV: {summary_path}")
print(f"Point-label CSV: {points_path}")
print(f"Plotted proto nodes: {sum(row['status'] == 'plotted' for row in summary_rows)} / {len(summary_rows)}")


Loading text encoder models in device: GPU


/home/xiaoyue/anaconda3/envs/llm_graph/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Saved proto semantic-label plots to: /home/xiaoyue/LiteSemRAG/cache/proto_description_label_plots/20260412_175422
Summary CSV: /home/xiaoyue/LiteSemRAG/cache/proto_description_label_plots/20260412_175422/summary.csv
Point-label CSV: /home/xiaoyue/LiteSemRAG/cache/proto_description_label_plots/20260412_175422/point_labels.csv
Plotted proto nodes: 2 / 74


In [ ]:
import time
correct = 0
mrr_sum = 0
start = time.time()
for sample_idx, sample in enumerate(samples[:500]):
    num_correct = 0
    #print(sample['question'])
    _, retrieved_chunk_node_id, cog = graph_database.multi_level_query(sample['question'], top_k_chunk=10, top_k_each_isolated_chunk=2, isolate_retrieve_mode='sequential', isolate_chunk_ratio=0.5,print_important_tokens=False)
    #rerank_chunks, retrieved_chunk_node_id = graph_database.broad_search_query(sample['question'],top_k=10)
    all_titles = []
    for idx in retrieved_chunk_node_id:
        all_titles.append(graph_database.chunk_nodes[idx].doc_node.doc_name)
    correct_titles = [documents[index]['title'] for index in sample['gold_doc_ids']]
    mrr = mrr_for_one_query_titles(all_titles, correct_titles, k=10)
    mrr_sum += mrr
    for answer in correct_titles:
        if answer in all_titles:
            correct = correct + 1
            num_correct += 1
    #print(f"Correct: {num_correct}")
    #print("=========================================================================")
    if num_correct < 2:
        print(f"question {sample_idx}:{sample['question']}, correct:{num_correct}, mrr:{mrr}")
    #print(f"question {sample_idx}:{sample['question']}, correct:{num_correct}")
print(correct/1000)
print(mrr_sum/500)
end = time.time()
print(f"运行时间：{end - start:.6f} 秒")

In [ ]:
inspect_index = 57
print(samples[inspect_index]['question'])
question = samples[inspect_index]['question']
retrieved_chunk, retrieved_chunk_node_id, cog = graph_database.multi_level_query(clean_text(question), top_k_chunk=10, isolate_retrieve_mode='sequential', isolate_chunk_ratio=0.2,print_important_tokens=True)
for index, chunk in enumerate(retrieved_chunk):
    print(f"Retrieved {index} :{chunk}")
print("====================================================================")
for index in samples[inspect_index]['gold_doc_ids']:
    print(documents[index]['text'])